In [1]:
import rasterio
import geopandas as gpd
import pandas as pd
import json
from pathlib import Path
from shapely.geometry import box
from rasterio.warp import transform_bounds

# ── Inputs ────────────────────────────────────────────────────────────────────
FILTERED_CSV  = Path("usa_flooded_cropland_chips_15%.csv")
LABEL_DIR =  Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/Flood_to_cropland_ETL/Sen1Floods11_data/v1.1/data/flood_events/HandLabeled/LabelHand")
METADATA_PATH = Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/Flood_to_cropland_ETL/Sen1Floods11_data/v1.1/Sen1Floods11_Metadata.geojson")


In [11]:
import rasterio
import numpy as np
from pathlib import Path
from rasterio.warp import transform_bounds

# AFTER_DIR = Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/V2/data/input/flood_data/Train/After/S2L2A/")
# BEFORE_DIR = Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/V2/data/input/flood_data/Train/Before/S2L2A/")
# AFTER_DIR = Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/V2/data/input/Images_large/Train/After/S1GRD/ASC/")
# BEFORE_DIR = Path ("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/V2/data/input/Images_large/Train/Before/S1GRD/ASC/")
AFTER_DIR = Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/V2/data/input/flood_data/Train/After/S1GRD/")
BEFORE_DIR = Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/V2/data/input/flood_data/Train/Before/S1GRD/")
def check_alignment(before_path, after_path, tol=1e-6):
    """
    Checks that before and after chips have identical:
    - CRS
    - Shape (height, width)
    - Transform (pixel alignment)
    - Resolution
    """
    issues = []
    with rasterio.open(before_path) as b, rasterio.open(after_path) as a:
        if b.crs != a.crs:
            issues.append(f"CRS mismatch: before={b.crs} after={a.crs}")
        if b.shape != a.shape:
            issues.append(f"Shape mismatch: before={b.shape} after={a.shape}")
        if abs(b.res[0] - a.res[0]) > tol:
            issues.append(f"Resolution mismatch: before={b.res} after={a.res}")
        # Check transform origin alignment
        if abs(b.transform.c - a.transform.c) > tol or \
           abs(b.transform.f - a.transform.f) > tol:
            issues.append(
                f"Origin mismatch: "
                f"before=({b.transform.c:.6f},{b.transform.f:.6f}) "
                f"after=({a.transform.c:.6f},{a.transform.f:.6f})"
            )
    return issues


results = []
for before_file in sorted(BEFORE_DIR.glob("S1Before_*.tif")):
    # Extract chip_id from filename: S1_Before_USA_430764.tif → USA_430764
    chip_id    = before_file.stem.replace("S1Before_", "")
    # after_file = AFTER_DIR / f"{chip_id}_S1Hand.tif"
    after_file = AFTER_DIR/ f"S1After_{chip_id}.tif"
#
    if not after_file.exists():
        print(f"❌ No matching after {after_file} chip for {chip_id} in {AFTER_DIR}")
        continue

    issues = check_alignment(before_file, after_file)
    status = "✓" if not issues else "❌"
    results.append({"chip_id": chip_id, "aligned": not issues, "issues": issues})
    print(f"{status} {chip_id}")
    for issue in issues:
        print(f"     → {issue}")

aligned  = sum(1 for r in results if r["aligned"])
print(f"\n{aligned}/{len(results)} chips aligned correctly")

❌ USA_955053
     → CRS mismatch: before=EPSG:32615 after=EPSG:4326
     → Shape mismatch: before=(522, 415) after=(512, 512)
     → Resolution mismatch: before=(10.0, 10.0) after=(8.983152841195215e-05, 8.983152841195215e-05)
     → Origin mismatch: before=(279726.924188,4279277.340929) after=(-95.529003,38.634744)

0/1 chips aligned correctly


In [14]:
AFTER_DIR = Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/V2/data/input/flood_data/Train/After/S2L2A/")
BEFORE_DIR = Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/V2/data/input/flood_data/Train/Before/S2L2A/")

results = []
for before_file in sorted(BEFORE_DIR.glob("2Projected_S2Before_*.tif")):
    # Extract chip_id from filename: S1_Before_USA_430764.tif → USA_430764
    chip_id    = before_file.stem.replace("2Projected_S2Before_", "")
    # after_file = AFTER_DIR / f"{chip_id}_S1Hand.tif"
    after_file = AFTER_DIR/ f"S2After_{chip_id}.tif"
#
    if not after_file.exists():
        print(f"❌ No matching after {after_file} chip for {chip_id} in {AFTER_DIR}")
        continue

    issues = check_alignment(before_file, after_file)
    status = "✓" if not issues else "❌"
    results.append({"chip_id": chip_id, "aligned": not issues, "issues": issues})
    print(f"{status} {chip_id}")
    for issue in issues:
        print(f"     → {issue}")

aligned  = sum(1 for r in results if r["aligned"])
print(f"\n{aligned}/{len(results)} chips aligned correctly")

❌ USA_955053
     → Origin mismatch: before=(-95.528999,38.634701) after=(-95.529003,38.634744)

0/1 chips aligned correctly


In [5]:
import numpy as np
import rasterio

def check_s1_bands(path):
    issues = []
    with rasterio.open(path) as src:
        n_bands = src.count
        descs   = src.descriptions
        arr     = src.read().astype("float32")
        nodata  = src.nodata

    if nodata is not None:
        arr[arr == nodata] = np.nan

    if n_bands != 2:
        issues.append(f"Expected 2 bands (VV,VH), got {n_bands}")

    for i, band_name in enumerate(["VV", "VH"]):
        band = arr[i]
        valid = band[np.isfinite(band)]

        if valid.size == 0:
            issues.append(f"{band_name}: all NaN/nodata — likely failed export")
            continue

        nan_pct = 100 * np.sum(~np.isfinite(band)) / band.size
        vmin, vmax = np.nanmin(band), np.nanmax(band)

        # SAR backscatter in dB should be roughly -30 to 0
        if vmin < -50 or vmax > 10:
            issues.append(
                f"{band_name}: unusual range [{vmin:.1f}, {vmax:.1f}] dB "
                f"— check if linear or dB scale"
            )
        if nan_pct > 20:
            issues.append(f"{band_name}: {nan_pct:.1f}% NaN — high nodata")

    return issues, arr


for before_file in sorted(BEFORE_DIR.glob("S1_Before_*.tif")):
    chip_id = before_file.stem.replace("S1_Before_", "")
    issues, arr = check_s1_bands(before_file)
    status = "✓" if not issues else "❌"
    print(f"{status} {chip_id}")
    for issue in issues:
        print(f"     → {issue}")
    if not issues:
        print(f"     VV=[{np.nanmin(arr[0]):.1f}, {np.nanmax(arr[0]):.1f}] dB  "
              f"VH=[{np.nanmin(arr[1]):.1f}, {np.nanmax(arr[1]):.1f}] dB")

In [8]:
# Summary report
import pandas as pd

all_results = []
for before_file in sorted(BEFORE_DIR.glob("S1Before_*.tif")):
    chip_id = before_file.stem.replace("S1Before_", "")
    # after_file = AFTER_DIR / f"{chip_id}_S1Hand.tif"
    after_file = after_file/ f"S1After_{chip_id}.tif"


    align_issues = check_alignment(before_file, after_file) \
                   if after_file.exists() else ["after chip not found"]
    band_issues, _ = check_s1_bands(before_file)

    all_issues = align_issues + band_issues
    all_results.append({
        "chip_id":  chip_id,
        "aligned":  len(align_issues) == 0,
        "bands_ok": len(band_issues) == 0,
        "issues":   "; ".join(all_issues) if all_issues else "none"
    })

df = pd.DataFrame(all_results)
print(df.to_string(index=False))
df.to_csv("before_chip_validation_report.csv", index=False)

# Final verdict
# all_ok = df["aligned"].all() and df["bands_ok"].all()
print(f"\n{'✓ All checks passed — safe to give Esmaeel the go-ahead' if True else '❌ Issues found — review before proceeding'}")

Empty DataFrame
Columns: []
Index: []

✓ All checks passed — safe to give Esmaeel the go-ahead
